## Prepare questions from PPMI for training GRPO

Created by: Shreyas & Sahana

Date: 07/15/2025

In [1]:
import pandas as pd
import json
import random
import string

In [2]:
directory_path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/ppmi"

In [3]:
with open(f"{directory_path}/PPMI_dictionary.json", 'r') as json_file:
    data_dict = json.load(json_file)

In [4]:
ppmi_path = '/projectnb/vkolagrp/datasets/PPMI/csv/raw/PPMI_Curated_Data_Cut_Public_20240729/20240703-Table 1.csv'
ppmi_df = pd.read_csv(ppmi_path, nrows=None)

In [5]:
def get_latest_visits(data):
    result = data.sort_values(by=['PATNO', 'YEAR'], ascending=[True, False])
    result = result.drop_duplicates(subset='PATNO', keep='first').reset_index(drop=True)
    return result

ppmi_latest = get_latest_visits(ppmi_df)
ppmi_latest['COHORT'] = 'PPMI'
ppmi_latest = ppmi_latest[(~ppmi_latest['cogstate'].isna()) & (~ppmi_latest['PRIMDIAG'].isna())].reset_index(drop=True)

In [6]:
# filter the ppmi_latest to turn any PRIMDIAG that is not in prim_diag_to_keep to 97.0
ppmi_latest['PRIMDIAG'] = ppmi_latest['PRIMDIAG'].apply(
    lambda x: x if len(ppmi_latest[ppmi_latest['PRIMDIAG'] == x]) > 15 else 97.0
)
prim_diag_to_keep = list(ppmi_latest['PRIMDIAG'].value_counts(dropna=False).keys())
ppmi_latest['PRIMDIAG'].value_counts(dropna=False)

PRIMDIAG
1.0     1373
17.0     878
25.0     867
97.0     119
7.0       40
23.0      19
24.0      19
Name: count, dtype: int64

In [7]:
prim_diag_to_keep

[1.0, 17.0, 25.0, 97.0, 7.0, 23.0, 24.0]

In [8]:
# edit the data dict for primdiag to only include the values we are keeping
data_dict['Diagnosis']['PRIMDIAG']['Values'] = {k: v for k, v in data_dict['Diagnosis']['PRIMDIAG']['Values'].items() if float(k) in prim_diag_to_keep}
data_dict['Diagnosis']['PRIMDIAG']['Values']


{'1': 'Idiopathic PD',
 '7': 'Essential tremor',
 '17': 'No PD nor other neurological disorder',
 '23': 'Prodromal non-motor PD',
 '24': 'Prodromal motor PD',
 '25': 'Prodromal Synucleinopathy',
 '97': 'Other neurological disorder(s)'}

In [9]:
def process_val(field, key, row, json_obj):
    '''
    Process a value from a field and add it to the json object
    '''
    # get the description
    description = data_dict[field][key]['Description']
    values = data_dict[field][key]['Values']

    # get the value from the row
    if isinstance(row[key], float):
        if pd.isna(row[key]):
            # print(f'row_val is nan, skipping')
            return
        # if its a float then convert to int
        row_val = str(int(row[key]))
    else:
        row_val = str(row[key])
        
    # if row_val is nan then skip
    try:
        if pd.isna(row_val) or row_val == 'nan':
            # print(f'row_val is nan, skipping')
            return

        # if values has '<UND>: Any text or numbers' then no need to look up
        if '<UND>: Any text or numbers' in values:
            assert len(values) == 1
        
        # if values is just a list not a dict then no need to look up either
        elif isinstance(values, list):
            # do nothing
            pass
            
        else:
            assert isinstance(values, dict)
            # look up the value in the row
            row_val = values[row_val]

        # if its not empty then add it to the json
        if pd.isna(row_val): pass
        elif row_val == '': pass
        elif row_val == 'nan': pass
        elif 'Unknown' in row_val: pass
        elif 'Not Reported' in row_val: pass
        else:
            # add to json
            # print(f'Adding {description} to json')
            json_obj[description] = row_val
    except Exception as e:
        raise Exception(f'KeyError: {e} for field: {field}, key: {key}, row_val: {row_val}\n dictionary: {data_dict[field][key]}')
    

def create_json(row, dict, mask=[]):
    '''
    Create a json object from a row and a dictionary
    mask is a list of keys to ignore

    pt info
        The dict has keys 'Demographics', 'Clinical',  'Milestones', 'Genetics', 'DATSCAN'
        also include field 'Stage_G' from 'NSD-ISS' key

    answers:
    'NSD-ISS' except 'Stage_G' has parkinsons scores like datscan and biochem assay and also motor symptom scales
    'Biologics' has neuropath like CSF and blood assays

    from 'Clinical' we can take the following:
        'PRIMDIAG' and 'OTHNEURO': {'Description': 'Comment for most likely Primary Diagnosis = "Other" (PRIMDIAG = 97)'
        'MCI_testscores' and  'cogstate' for cognitive status

    '''
    json_obj = {}
    # start with demographics
    json_obj['Demographics'] = {}
    for key in dict['Demographics']:
        if key in mask or key == "age":
            continue
        else:
            process_val('Demographics', key, row, json_obj['Demographics'])
    
    # add clinical except for 'PRIMDIAG' and 'OTHNEURO' and MCI_testscores and 'cogstate'
    json_obj['Clinical'] = {}
    for key in dict['Clinical']:
        if key in mask:
            continue
        process_val('Clinical', key, row, json_obj['Clinical'])
    
    # add Stage_subpark (from NSD-IS) to clinical
    if 'Stage_subpark' not in mask:
        process_val('NSD-ISS', 'Stage_subpark', row, json_obj['Clinical'])
    # add Stage_partial_UPDRS1 to clinical
    if 'Stage_partial_UPDRS1' not in mask:
        process_val('NSD-ISS', 'Stage_partial_UPDRS1', row, json_obj['Clinical'])
    # add Stage_PDTreat to clinical
    if 'Stage_PDTreat' not in mask:
        process_val('NSD-ISS', 'Stage_PDTreat', row, json_obj['Clinical'])
        
    
    # add the biologics  "hemohi", "urate",
    # "total_di_18_1_BMP", "total_di_22_6_BMP", "_2_2__di_22_6_BMP",
    # "NFL_CSF", "nfl_serum" all to clinical
    for key in dict['Biologics']:
        if key in mask:
            continue
        if key in ['hemohi', 'urate', 'total_di_18_1_BMP', 'total_di_22_6_BMP', '_2_2__di_22_6_BMP', 'NFL_CSF', 'nfl_serum', "asyn", "abeta"]:
            process_val('Biologics', key, row, json_obj['Clinical'])

    # add the Milestones
    json_obj['Milestones'] = {}
    for key in dict['Milestones']:
        if key in mask:
            continue
        process_val('Milestones', key, row, json_obj['Milestones'])
    
    # add the Genetics
    json_obj['Genetics'] = {}
    for key in dict['Genetics']:
        if key in mask:
            continue
        process_val('Genetics', key, row, json_obj['Genetics'])
        
    # add 'NSD-ISS' stage g
    if 'Stage_G' not in mask:
        process_val('NSD-ISS', 'Stage_G', row, json_obj['Genetics']) # this row doesn't even matter tbh because none of the rows have this info

    

    # add the DATSCAN
    json_obj['DATSCAN'] = {}
    for key in dict['DATSCAN']:
        if key in mask:
            continue
        process_val('DATSCAN', key, row, json_obj['DATSCAN'])
    
    # remove (Note: Results not shown for Prodromal cohort in Public Data Cut) from all the keys in 'DATSCAN'
    update_json = {}
    for key in json_obj['DATSCAN']:
        if 'Results not shown for Prodromal cohort in Public Data Cut' in key:
            new_key = key.replace(' (Note: Results not shown for Prodromal cohort in Public Data Cut)', '')
            update_json[new_key] = json_obj['DATSCAN'][key]
        else:
            update_json[key] = json_obj['DATSCAN'][key]
    json_obj['DATSCAN'] = update_json
    
    json_obj = {k: v for k, v in json_obj.items() if v}

    return json_obj

## GRPO training data

In [10]:
grpo_vars = [
    'Stage_S',
    'NSD_Status',
    'Stage_D',
    'cogstate',
    'PRIMDIAG',
]

In [11]:
def get_options(values_dict, row, var, seed=None):
    options_texts = list(values_dict.values())
    rng = random.Random(seed if seed is not None else row.name)
    shuffled = options_texts.copy()
    rng.shuffle(shuffled)
    letters = list(string.ascii_uppercase[:len(shuffled)])
    options = "\n".join(f"{ltr}. {opt}" for ltr, opt in zip(letters, shuffled))

    # 3️⃣ Determine correct answer letter
    # print(values_dict, row[var])
    correct_text = values_dict.get(str(int(row[var])), None)
    # try:
    #     if np.isclose(float(k), float(row[var])):
    #         ground_truth = L
    # except ValueError:
    #     if k == row[var]:
    #         ground_truth = L

    # if correct_text in shuffled:
    #     ground_truth = letters[shuffled.index(correct_text)]
    # else:
    #     ground_truth = None  # fallback for unrecognized codes
        
    ground_truth = letters[shuffled.index(correct_text)]
        
    return options, ground_truth, correct_text

In [12]:
def generate_grpo_question_data(row, values_dict, data_dict, var, questions, mask=[], Q_TYPE=None):
    import string
    import numpy as np
    '''
    returns: patient_summary, question, options, ground_truth, ground_truth, Q_TYPE
    - question: randomly sample a questin from the questions list
    - options: multiple-choice format of the var derived from the values_dict like A. Amnestic MCI- single domain\nB. Amnestic MCI- multiple domain\nC. etc,
    - ground_truth: the correct letter choice like 'A' or 'B'
    - Q_TYPE: if Q_type is none jsut use the var name, otherwise use the Q_TYPE provided

    - patient_summary: a json object created from the row and the values_dict, with the right stuff masked
    '''
    question = questions.sample().iloc[0]

    assert isinstance(values_dict, dict), f'values_dict must be a dict, got {type(values_dict)}'
    # turn the values_dict into a multiple-choice format
    # options = {}
    # ground_truth = None
    # for (k, v), L in zip(values_dict.items(), string.ascii_uppercase):
    #     # options += f"{L}. {v}\n"
    #     options[L] = v
    #     try:
    #         if np.isclose(float(k), float(row[var])):
    #             ground_truth = L
    #     except ValueError:
    #         if k == row[var]:
    #             ground_truth = L
    
    options, ground_truth, ground_truth_text = get_options(values_dict, row, var)
    
    # print(options, ground_truth, ground_truth_text)
    # raise ValueError
    assert options != '', f'options cannot be empty, got {options}'
    assert ground_truth != None, f'ground_truth cannot be empty, got row val: {row[var]} and options: {options}'
    if Q_TYPE is None:
        Q_TYPE = var
    
    # create the patient_summary
    patient_summary = create_json(row, data_dict, mask=mask)

    return patient_summary, question, options, ground_truth, ground_truth_text, Q_TYPE


In [13]:
# for each selected row, pass it to generate_grpo_question_data to add the information to ppmi_latest
# however, we also want to not repeat questions for the same PATNO, so we will keep track of the questions already asked for each PATNO
# we will use a maximum of 500 PATNOs per question type, so we will limit the number of rows in ppmi_latest to 500 for each question type 
# also make sure that we are only using a PATNO for a question type if it actually has the data, i.e. is not NaN for the var

max_per_question_type = 500

stage_s_questions = [
    "Identify the alpha-synuclein seed amplification assay (alpha-syn SAA) result for this subject.",
    "Determine the alpha-synuclein seed amplification assay (alpha-syn SAA) status from the provided information.",
    "What is the alpha-synuclein seed amplification assay (alpha-syn SAA) result for this individual?",
    "Based on the text, identify the subject's alpha-synuclein seed amplification assay (alpha-syn SAA) outcome.",
    "Find and report the alpha-synuclein seed amplification assay (alpha-syn SAA) result for this patient."
]


stage_s_values_dict = data_dict['NSD-ISS']['Stage_S']['Values']
diag_desc = data_dict['NSD-ISS']['Stage_S']['Description']
assert isinstance(stage_s_values_dict, dict), f'stage_s_values_dict must be a dict, got {type(stage_s_values_dict)}'
sub = ppmi_latest.dropna(subset=['Stage_S']) # filter missing
print(f"Total rows: {len(sub)}")
if len(sub) > max_per_question_type:
    sub = sub.sample(n=max_per_question_type, random_state=42)  # sample 500 rows if more than 500
for i, row in sub.iterrows():
    # generate the question data
    patient_summary, question, options, ground_truth, ground_truth_text, Q_TYPE = generate_grpo_question_data(
        row=row,
        values_dict=stage_s_values_dict,
        data_dict=data_dict,
        var='Stage_S',
        questions=pd.Series(stage_s_questions),
        mask=['asyn'], # mask the asyn value
        Q_TYPE='a-syn SAA status'
    )
    diag_summary = {
        diag_desc: ground_truth_text
    }
    # add to ppmi_latest
    ppmi_latest.at[i, 'patient_summary'] = json.dumps(patient_summary)
    ppmi_latest.at[i, 'diag_summary'] = json.dumps(diag_summary)
    ppmi_latest.at[i, 'question'] = question
    ppmi_latest.at[i, 'options'] = options
    ppmi_latest.at[i, 'ground_truth'] = ground_truth
    ppmi_latest.at[i, 'ground_truth_text'] = ground_truth_text
    ppmi_latest.at[i, 'Q_TYPE'] = Q_TYPE

print(f"Added {ppmi_latest['Q_TYPE'].value_counts().get('a-syn SAA status', 0)} rows for Stage_S question.")

Total rows: 1173
Added 500 rows for Stage_S question.


In [14]:
NSD_status_questions = [
    "Identify the subject's neuronal alpha-synuclein disease (NSD) status.",
    "Determine whether the individual is classified as neuronal alpha-synuclein disease (NSD) based on the provided information.",
    "What is the neuronal alpha-synuclein disease (NSD) status for this participant?",
    "Based on the available data, identify if the patient has neuronal alpha-synuclein disease (NSD) or not.",
    "Using the given information, determine the subject's neuronal alpha-synuclein disease (NSD) classification."
]


NSD_status_values_dict = data_dict['NSD-ISS']['NSD_Status']['Values']
diag_desc = data_dict['NSD-ISS']['NSD_Status']['Description']
assert isinstance(NSD_status_values_dict, dict), f'NSD_status_values_dict must be a dict, got {type(NSD_status_values_dict)}'
sub = ppmi_latest.dropna(subset=['NSD_Status'])  # filter missing
sub = sub[sub['Q_TYPE'].isna()] # filter for only the rows wher Q_TYPE is missing
print(f"Total rows: {len(sub)}")
if len(sub) > max_per_question_type:
    sub = sub.sample(n=max_per_question_type, random_state=42)  # sample 500 rows if more than 500
for i, row in sub.iterrows():
    # generate the question data
    patient_summary, question, options, ground_truth, ground_truth_text, Q_TYPE = generate_grpo_question_data(
        row=row,
        values_dict=NSD_status_values_dict,
        data_dict=data_dict,
        var='NSD_Status',
        questions=pd.Series(NSD_status_questions),
        mask=['asyn'],  # mask the asyn value
        Q_TYPE='NSD Status'
    )
    diag_summary = {
        diag_desc: ground_truth_text
    }
    # add to ppmi_latest
    ppmi_latest.at[i, 'patient_summary'] = json.dumps(patient_summary)
    ppmi_latest.at[i, 'diag_summary'] = json.dumps(diag_summary)
    ppmi_latest.at[i, 'question'] = question
    ppmi_latest.at[i, 'options'] = options
    ppmi_latest.at[i, 'ground_truth'] = ground_truth
    ppmi_latest.at[i, 'ground_truth_text'] = ground_truth_text
    ppmi_latest.at[i, 'Q_TYPE'] = Q_TYPE
print(f"Added {ppmi_latest['Q_TYPE'].value_counts().get('NSD Status', 0)} rows for NSD_Status question.")


Total rows: 685
Added 500 rows for NSD_Status question.


In [15]:
# now Stage_D question --------------
dat_scan_questions = [
    "Classify if the patient is DatScan positive based on the provided information.",
    "Using the given data, determine whether the patient is DatScan positive.",
    "Analyze the provided information to classify the patient's DatScan status.",
    "Based on the information given, classify whether the patient is DatScan positive.",
    "From the provided patient details, identify if the patient is DatScan positive."
]
dat_scan_values_dict = data_dict['NSD-ISS']['Stage_D']['Values']
diag_desc = data_dict['NSD-ISS']['Stage_D']['Description']
assert isinstance(dat_scan_values_dict, dict), f'dat_scan_values_dict must be a dict, got {type(dat_scan_values_dict)}'
sub = ppmi_latest.dropna(subset=['Stage_D'])  # filter missing
sub = sub[sub['Q_TYPE'].isna()]  # filter for only the rows where Q_TYPE is missing
print(f"Total rows: {len(sub)}")
if len(sub) > max_per_question_type:
    sub = sub.sample(n=max_per_question_type, random_state=42)  # sample 500 rows if more than 500
for i, row in sub.iterrows():
    # generate the question data
    patient_summary, question, options, ground_truth, ground_truth_text, Q_TYPE = generate_grpo_question_data(
        row=row,
        values_dict=dat_scan_values_dict,
        data_dict=data_dict,
        var='Stage_D',
        questions=pd.Series(dat_scan_questions),
        mask=list(data_dict.get('DATSCAN').keys()),  # mask all DATSCAN features
        Q_TYPE='DATSCAN'
    )
    diag_summary = {
        diag_desc: ground_truth_text
    }
    # add to ppmi_latest
    ppmi_latest.at[i, 'patient_summary'] = json.dumps(patient_summary)
    ppmi_latest.at[i, 'diag_summary'] = json.dumps(diag_summary)
    ppmi_latest.at[i, 'question'] = question
    ppmi_latest.at[i, 'options'] = options
    ppmi_latest.at[i, 'ground_truth'] = ground_truth
    ppmi_latest.at[i, 'ground_truth_text'] = ground_truth_text
    ppmi_latest.at[i, 'Q_TYPE'] = Q_TYPE
print(f"Added {ppmi_latest['Q_TYPE'].value_counts().get('DATSCAN', 0)} rows for Stage_D question.")


Total rows: 311
Added 311 rows for Stage_D question.


In [16]:
# now cogstate question --------------
cogstate_questions = [
    "Using the information provided, determine the patient's cognitive status.",
    "Identify the patient's cognitive status based on the given information.",
    "Based on the provided clinical details, classify the patient's cognitive status.",
    "From the information available, determine the correct cognitive status for the patient.",
    "Analyze the patient's information and identify their cognitive status."
]

cogstate_values_dict = data_dict['Diagnosis']['cogstate']['Values']
diag_desc = data_dict['Diagnosis']['cogstate']['Description']
assert isinstance(cogstate_values_dict, dict), f'cogstate_values_dict must be a dict, got {type(cogstate_values_dict)}'
sub = ppmi_latest.dropna(subset=['cogstate'])  # filter missing
sub = sub[sub['Q_TYPE'].isna()]  # filter for only the rows where Q_TYPE is missing
print(f"Total rows: {len(sub)}")
if len(sub) > max_per_question_type:
    sub = sub.sample(n=max_per_question_type, random_state=42)  # sample 500 rows if more than 500
for i, row in sub.iterrows():
    # generate the question data
    patient_summary, question, options, ground_truth, ground_truth_text, Q_TYPE = generate_grpo_question_data(
        row=row,
        values_dict=cogstate_values_dict,
        data_dict=data_dict,
        var='cogstate',
        questions=pd.Series(cogstate_questions),
        mask=[],  # no mask for cogstate
        Q_TYPE='Cognitive status'
    )
    diag_summary = {
        diag_desc: ground_truth_text
    }
    # add to ppmi_latest
    ppmi_latest.at[i, 'patient_summary'] = json.dumps(patient_summary)
    ppmi_latest.at[i, 'diag_summary'] = json.dumps(diag_summary)
    ppmi_latest.at[i, 'question'] = question
    ppmi_latest.at[i, 'options'] = options
    ppmi_latest.at[i, 'ground_truth'] = ground_truth
    ppmi_latest.at[i, 'ground_truth_text'] = ground_truth_text
    ppmi_latest.at[i, 'Q_TYPE'] = Q_TYPE
print(f"Added {ppmi_latest['Q_TYPE'].value_counts().get('Cognitive status', 0)} rows for cogstate question.")


Total rows: 2004
Added 500 rows for cogstate question.


In [17]:
# now PRIMDIAG question --------------
sub = ppmi_latest.dropna(subset=['PRIMDIAG'])  # filter missing
sub = sub[sub['Q_TYPE'].isna()]  # filter for only the rows where Q_TYPE is missing
print(f"There are {len(sub)} rows left for PRIMDIAG question.")

There are 1504 rows left for PRIMDIAG question.


In [18]:
# if len(sub) > max_per_question_type:
#     sub = sub.sample(n=max_per_question_type, random_state=42)

In [19]:
# no need to sample here, we want all the remaining rows with PRIMDIAG
primdiag_questions = [
    "What is the primary diagnosis of the patient based on the provided data?",
    "Using the available information, classify the patient's primary diagnosis.",
    "Determine the primary diagnosis of the individual from the given patient infromation.",
    "Based on the provided information, what is the patient's primary diagnosis?",
    "Identify the subject's primary diagnosis using the provided information."
]
primdiag_values_dict = data_dict['Diagnosis']['PRIMDIAG']['Values']
diag_desc = data_dict['Diagnosis']['PRIMDIAG']['Description']
print(f"PRIMDIAG values_dict: {primdiag_values_dict}")
assert isinstance(primdiag_values_dict, dict), f'primdiag_values_dict must be a dict, got {type(primdiag_values_dict)}'

for i, row in sub.iterrows():
    # generate the question data
    patient_summary, question, options, ground_truth, ground_truth_text, Q_TYPE = generate_grpo_question_data(
        row=row,
        values_dict=primdiag_values_dict,
        data_dict=data_dict,
        var='PRIMDIAG',
        questions=pd.Series(primdiag_questions),
        mask=[],  # no mask
        Q_TYPE='Primary etiology'
    )
    diag_summary = {
        diag_desc: ground_truth_text
    }
    # add to ppmi_latest
    ppmi_latest.at[i, 'patient_summary'] = json.dumps(patient_summary)
    ppmi_latest.at[i, 'diag_summary'] = json.dumps(diag_summary)
    ppmi_latest.at[i, 'question'] = question
    ppmi_latest.at[i, 'options'] = options
    ppmi_latest.at[i, 'ground_truth'] = ground_truth
    ppmi_latest.at[i, 'ground_truth_text'] = ground_truth_text
    ppmi_latest.at[i, 'Q_TYPE'] = Q_TYPE
    
print(f"Added {ppmi_latest['Q_TYPE'].value_counts().get('Primary etiology', 0)} rows for PRIMDIAG question.")



PRIMDIAG values_dict: {'1': 'Idiopathic PD', '7': 'Essential tremor', '17': 'No PD nor other neurological disorder', '23': 'Prodromal non-motor PD', '24': 'Prodromal motor PD', '25': 'Prodromal Synucleinopathy', '97': 'Other neurological disorder(s)'}
Added 1504 rows for PRIMDIAG question.


In [20]:
ppmi_latest[ppmi_latest['Q_TYPE'] == "Primary etiology"]['PRIMDIAG'].value_counts()

PRIMDIAG
17.0    653
25.0    643
1.0      86
97.0     70
7.0      25
23.0     15
24.0     12
Name: count, dtype: int64

In [21]:
ppmi_latest.drop(['question', 'options', 'ground_truth', 'ground_truth_text', 'Q_TYPE'], axis=1).to_csv(f"{directory_path}/train.csv", index=False)

In [22]:
ppmi_latest['ID'] = ppmi_latest['PATNO']
ppmi_latest = ppmi_latest[[
    'ID', 'patient_summary', 'diag_summary', 'question', 'options', 'ground_truth', 'ground_truth_text', 'COHORT', 'Q_TYPE'
]]

In [23]:
ppmi_latest.head()

,ID,patient_summary,diag_summary,question,options,ground_truth,ground_truth_text,COHORT,Q_TYPE
0,3000,"{""Demographics"": {""Age at Visit"": ""79"", ""Sex a...","{""Most likely Primary Diagnosis"": ""No PD nor o...",Determine the primary diagnosis of the individ...,A. Prodromal motor PD\nB. No PD nor other neur...,B,No PD nor other neurological disorder,PPMI,Primary etiology
1,3001,"{""Demographics"": {""Age at Visit"": ""77"", ""Sex a...","{""Indicator for neuronal \u03b1-synuclein dise...",Determine whether the individual is classified...,A. NSD\nB. Not NSD,A,NSD,PPMI,NSD Status
2,3002,"{""Demographics"": {""Age at Visit"": ""78"", ""Sex a...","{""Visit-level DAT SBR status - indicates wheth...",Analyze the provided information to classify t...,A. Yes\nB. No,A,Yes,PPMI,DATSCAN
3,3003,"{""Demographics"": {""Age at Visit"": ""69"", ""Sex a...","{""Subject-level a-syn SAA status (Note: Result...","Based on the text, identify the subject's alph...",A. MSA-like\nB. Negative\nC. Inconclusive\nD. ...,D,Positive,PPMI,a-syn SAA status
4,3004,"{""Demographics"": {""Age at Visit"": ""72"", ""Sex a...","{""Most likely Primary Diagnosis"": ""No PD nor o...",Determine the primary diagnosis of the individ...,A. Prodromal Synucleinopathy\nB. Prodromal mot...,F,No PD nor other neurological disorder,PPMI,Primary etiology


In [30]:
json.loads(ppmi_latest.iloc[0]['patient_summary'])

{'Demographics': {'Age at Visit': '79',
  'Sex at birth': 'Female',
  'Years of Education capped at 20': '18',
  'Race': 'White',
  'Indicator for Hispanic/Latino ethnicity': 'No',
  'Indicator for Ashkenazi Jewish descent': 'No',
  'Indicator for African Berber descent': 'No',
  'Indicator for Basque descent': 'No',
  'Family History of PD - 3 categories.  NOTE: First degree family history of PD was an exclusion criterion for Healthy Controls so HCs should not be used for comparison purposes.': 'No Family w/PD',
  'Family History of PD - Binary': 'No Family w/PD',
  'Handedness': 'Right'},
 'Clinical': {'Body Mass Index': '52',
  'MOCA Score (adjusted for education)': '26',
  'Benton Judgement of Line Orientation Score': '14',
  'Total Clock Drawing Score': '7',
  'HVLT Discrimination Recognition': '7',
  'HVLT Immediate/Total Recall': '21',
  'HVLT Retention': '0',
  'HVLT False Alarms': '0',
  'HVLT Delayed Recall ': '4',
  'HVLT Delayed Recognition': '8',
  'Lexical Fluency Score':

In [24]:
ppmi_latest['Q_TYPE'].value_counts()

Q_TYPE
Primary etiology    1504
NSD Status           500
a-syn SAA status     500
Cognitive status     500
DATSCAN              311
Name: count, dtype: int64

In [25]:
# json.loads(ppmi_latest.iloc[3]['patient_summary'])

In [26]:
ppmi_latest.to_csv(f"{directory_path}/training_data/training_data_grpo/train_with_questions.csv", index=False)

In [27]:
question_df = ppmi_latest.groupby(['question', 'Q_TYPE']).size().reset_index(name='count').sort_values(by=['Q_TYPE', 'count'])

In [28]:
# Save as CSV
csv_path = f"{directory_path}/training_data/training_data_grpo/train_question_counts.csv"
question_df.to_csv(csv_path, index=False)